# Sila na koljeno — promjena smjera strujanja

**Poglavlje U11: Količina gibanja i sile strujanja**

Ovaj interaktivni prikaz nadopunjuje izvod sile fluida na horizontalno koljeno. Mijenjanjem kuta zakretanja, protoka i promjera cijevi prati se vektorska sila koju nosač mora preuzeti.

## Cilj

Kada fluid u koljenu mijenja smjer, na konstrukciju djeluje sila koja proizlazi iz promjene količine gibanja i iz tlakova na ulaznom i izlaznom presjeku. Prikaz omogućuje:

1. mijenjanje kuta zakretanja koljena $\beta$;
2. mijenjanje volumenskog protoka $Q$;
3. mijenjanje promjera cijevi $D$ (isti na ulazu i izlazu);
4. praćenje komponenti sile $F_x$, $F_y$ i ukupne rezultante.

## Pretpostavke modela

- horizontalno koljeno (težina fluida zanemarena);
- isti promjer cijevi na ulazu i izlazu ($D_1 = D_2 = D$);
- jednodimenzijski profil brzina u presjecima;
- jednak manometarski tlak na ulazu i izlazu ($p_1 = p_2 = p$);
- voda gustoće $\rho = 998$ kg/m³.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Layout

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 10

## Računski model

Iz jednadžbe količine gibanja primijenjene na kontrolni volumen koji obuhvaća cijelo koljeno, ulazna strana u smjeru osi $x$ i izlazna strana pod kutem $\beta$ od osi $x$:

$$F_x = (\rho Q v + p A)(1 - \cos\beta),$$
$$F_y = (\rho Q v + p A)\sin\beta,$$

gdje je $v = Q/A$. Iznos rezultante i kut prema osi $x$:

$$F_R = \sqrt{F_x^2 + F_y^2}, \qquad \tan\varphi = \frac{F_y}{F_x}.$$

Ovo je sila fluida na koljeno; nosač mora preuzeti jednaku silu suprotnog smjera.

In [ ]:
RHO = 998.0
P_REL = 200_000.0  # manometarski tlak (Pa)

def koljeno(beta_deg, Q_Lpsek, D_mm):
    beta = np.radians(beta_deg)
    D = D_mm / 1000.0
    Q = Q_Lpsek / 1000.0  # m^3/s
    A = np.pi * D**2 / 4
    v = Q / A
    F_inten = RHO * Q * v + P_REL * A
    F_x = F_inten * (1 - np.cos(beta))
    F_y = F_inten * np.sin(beta)
    F_R = np.sqrt(F_x**2 + F_y**2)
    phi = np.degrees(np.arctan2(F_y, F_x))
    return {'F_x': F_x, 'F_y': F_y, 'F_R': F_R, 'phi': phi,
             'v': v, 'F_inten': F_inten}

## Interaktivni prikaz

Klizačima u nastavku biraju se kut zakretanja, protok i promjer cijevi. Prikaz pokazuje shemu koljena u horizontalnoj ravnini s vektorima brzina ulaza i izlaza te rezultantnom silom na konstrukciju.

In [ ]:
def koljeno_prikaz(beta_deg, Q_Lpsek, D_mm):
    r = koljeno(beta_deg, Q_Lpsek, D_mm)
    beta = np.radians(beta_deg)

    fig, ax = plt.subplots(figsize=(8, 7))

    # Ulazna cijev (po osi x prema točki 0,0)
    L_cijevi = 0.8
    polumjer = D_mm / 2000.0  # za prikaz
    ax.plot([-L_cijevi, 0], [polumjer, polumjer],
             color='#1565c0', lw=2)
    ax.plot([-L_cijevi, 0], [-polumjer, -polumjer],
             color='#1565c0', lw=2)

    # Izlazna cijev (pod kutem beta od osi x)
    x_kraj = L_cijevi * np.cos(beta)
    y_kraj = L_cijevi * np.sin(beta)
    # Perpendikularne offsete
    dx = -polumjer * np.sin(beta)
    dy = polumjer * np.cos(beta)
    ax.plot([0 + dx, x_kraj + dx], [0 + dy, y_kraj + dy],
             color='#1565c0', lw=2)
    ax.plot([0 - dx, x_kraj - dx], [0 - dy, y_kraj - dy],
             color='#1565c0', lw=2)

    # Spojni dio koljena
    theta_spoj = np.linspace(np.pi/2, np.pi/2 - beta + np.pi, 30)
    ax.fill_between([], [], [])

    # Strelica ulazne brzine
    ax.annotate('', xy=(0, 0), xytext=(-0.5, 0),
                 arrowprops=dict(arrowstyle='->',
                                   color='#1565c0', lw=3))
    ax.text(-0.55, 0.08, f'$v_1$ = {r["v"]:.2f} m/s',
             color='#1565c0', fontsize=10)

    # Strelica izlazne brzine
    ax.annotate('', xy=(0.5*np.cos(beta), 0.5*np.sin(beta)),
                 xytext=(0, 0),
                 arrowprops=dict(arrowstyle='->',
                                   color='#1565c0', lw=3))
    ax.text(0.55*np.cos(beta) + 0.05, 0.55*np.sin(beta),
             f'$v_2$ = {r["v"]:.2f} m/s',
             color='#1565c0', fontsize=10)

    # Vektor rezultante sile (skala)
    skala = 0.6 / max(r['F_R'], 1)
    Lx = r['F_x'] * skala
    Ly = r['F_y'] * skala
    ax.annotate('', xy=(Lx, Ly), xytext=(0, 0),
                 arrowprops=dict(arrowstyle='->',
                                   color='#c62828', lw=3))
    ax.text(Lx + 0.05, Ly + 0.05,
             f'$F_R$ = {r["F_R"]/1000:.2f} kN\n'
             f'$\\varphi$ = {r["phi"]:.1f}°',
             color='#c62828', fontsize=11)

    # Os x i y
    ax.axhline(0, color='gray', lw=0.5, alpha=0.5)
    ax.axvline(0, color='gray', lw=0.5, alpha=0.5)
    ax.text(1.3, -0.05, 'x', color='gray', fontsize=10)
    ax.text(-0.05, 1.3, 'y', color='gray', fontsize=10)

    ax.set_xlim(-1.0, 1.5)
    ax.set_ylim(-0.5, 1.5)
    ax.set_aspect('equal')
    ax.set_title(
        f'$\\beta$ = {beta_deg:.0f}°,  '
        f'$Q$ = {Q_Lpsek:.1f} L/s,  $D$ = {D_mm:.0f} mm\n'
        f'$F_x$ = {r["F_x"]/1000:.2f} kN,  '
        f'$F_y$ = {r["F_y"]/1000:.2f} kN'
    )
    ax.grid(ls=':', alpha=0.5)

    plt.tight_layout()
    plt.show()


interact(
    koljeno_prikaz,
    beta_deg=FloatSlider(min=15, max=170, step=5, value=90,
                          description='$\\beta$ (°)',
                          layout=Layout(width='420px')),
    Q_Lpsek=FloatSlider(min=1, max=50, step=1, value=20,
                         description='$Q$ (L/s)',
                         layout=Layout(width='420px')),
    D_mm=FloatSlider(min=50, max=250, step=10, value=120,
                      description='$D$ (mm)',
                      layout=Layout(width='420px'))
);

## Pitanja za istraživanje

1. **Pravokutno koljeno.** Pri $\beta = 90°$, koje su vrijednosti $F_x$ i $F_y$? Pod kojim kutom djeluje rezultanta i kako mora biti orijentiran nosač?

2. **Potpuni U-okret.** Pri $\beta \to 180°$, što se događa s $F_x$ i $F_y$? Zašto je to najveća moguća sila pri zadanom $Q$ i $D$?

3. **Tlakni i impulsni doprinos.** Provjeri za nekoliko kombinacija parametara koliki je relativni udjel $\rho Q v$ (impulsno) prema $p A$ (tlačno) u ukupnom članu $F_{int}$. Kada dominira impulsni dio, a kada tlačni?

4. **Skala s promjerom.** Pri konstantnom $Q$, kako $D$ utječe na ukupnu silu? Postoji li promjer pri kojem je sila minimalna?

## Veza s teorijom poglavlja

Ovaj prikaz materijalizira primjenu zakona količine gibanja iz poglavlja U11 na klasičnom inženjerskom problemu — sili na koljeno cjevovoda. Promjena smjera strujanja stvara silu na konstrukciju neovisno o gubicima u zavoju. U realnim sustavima gubici (poglavlje U10) doprinose razlici tlakova između ulaza i izlaza koljena, ali osnovna struktura sile ostaje ista.